# **QUANTIZATION PIPELINE**

### **Section 1: Fake Quantization Utilities**

Simulate uniform integer quantization on a tensor.
This mimics what happens when weights are stored in INT8/INT4 format: the continuous values are discretized to a fixed number of levels. Used during sensitivity analysis to measure each layer's quantization error without actually converting the model.

- Args:
    - tensor: Input tensor (e.g., Conv2d weights)
    - bits: Number of bits (4, 8, 16). 16+ returns tensor unchanged.
    - symmetric: Use symmetric quantization (range [-2^(b-1)+1, 2^(b-1)-1])
    - per_channel: Quantize each output channel independently (better accuracy)
    - channel_axis: Axis along which channels are defined (0 for Conv2d weights)

- Returns:
    - Dequantized tensor (float values restricted to discrete levels)

In [3]:
import torch

def fake_quantize_tensor(
    tensor: torch.Tensor,
    bits: int = 8,
    symmetric: bool = True,
    per_channel: bool = False,
    channel_axis: int = 0,
) -> torch.Tensor:
    if bits >= 16:
        return tensor

    if per_channel and tensor.dim() > 1:
        # Per-channel quantization: each output channel gets its own scale
        moved = tensor.transpose(channel_axis, 0)
        original_shape = moved.shape
        flat = moved.reshape(original_shape[0], -1)

        if symmetric:
            qmax = 2 ** (bits - 1) - 1
            max_per_channel = flat.abs().amax(dim=1)
            scales = (max_per_channel / qmax).clamp(min=1e-8)
            scales = scales.view(-1, 1)
            q = torch.round(flat / scales).clamp(-qmax, qmax)
            dequant = q * scales
        else:
            qmin, qmax = 0, 2 ** bits - 1
            t_min = flat.amin(dim=1, keepdim=True)
            t_max = flat.amax(dim=1, keepdim=True)
            scales = ((t_max - t_min) / (qmax - qmin)).clamp(min=1e-8)
            zero_points = torch.round(qmin - t_min / scales)
            q = torch.round(flat / scales + zero_points).clamp(qmin, qmax)
            dequant = (q - zero_points) * scales

        dequant = dequant.reshape(original_shape)
        return dequant.transpose(0, channel_axis)

    # Per-tensor quantization
    if symmetric:
        qmax = 2 ** (bits - 1) - 1
        scale = (tensor.abs().max() / qmax).clamp(min=1e-8)
        q = torch.round(tensor / scale).clamp(-qmax, qmax)
        return q * scale
    else:
        qmin, qmax = 0, 2 ** bits - 1
        t_min, t_max = tensor.min(), tensor.max()
        scale = ((t_max - t_min) / (qmax - qmin)).clamp(min=1e-8)
        zero_point = torch.round(qmin - t_min / scale)
        q = torch.round(tensor / scale + zero_point).clamp(qmin, qmax)
        return (q - zero_point) * scale

### **Section 2: Per-Layer Sensitivity Analysis**

It measures how quantizing each Conv2d layer individually affects detection accuracy, with special attention to small-target AP.
Since not all layers are equally sensitive to quantization, layers processing high-resolution feature maps (P2/P3 strides) that encode fine spatial details needed for small drone detection are disproportionately affected by INT8/INT4 quantization.

This analysis identifies which layers should retain higher precision in a mixed-precision scheme.

**Methodology:**
1. Load the FP32-trained YOLOv11n model
2. For each Conv2d layer, replace its weights with fake-quantized weights
3. Run validation and measure mAP, mAP@0.5, and small-object AP
4. Restore original weights
5. Sensitivity score = baseline_AP - quantized_AP (higher = more sensitive)

The small-object AP is the localization sensitivity metric, which measures how well the model detects drones smaller than 32x32 pixels.

The discrimination sensitivity metric measures how quantization affects the drone-vs-bird decision boundary. For each layer, we track per-class
AP and the off-diagonal entries of the confusion matrix. A layer with high discrimination sensitivity is one where quantizing it causes
drones to be misclassified as birds (or vice versa) — these are the layers that carry the fine-grained discriminative features.

Together, these two sensitivity dimensions reveal that the layers most critical for small-target localization are not always the same as those most critical for drone-vs-bird discrimination. This separation is the key insight enabling a more principled mixed-precision allocation.

In [14]:
from typing import List, Optional, Dict, Tuple
import torch.nn as nn
import json
from tqdm import tqdm

class SensitivityAnalyzer:
    def __init__(
        self,
        model_path: str,
        data_yaml: str,
        imgsz: int = 640,
        small_target_threshold: int = 32,
        confusion_classes: Optional[List[str]] = None,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
    ):
        from ultralytics import YOLO

        self.model_path = model_path
        self.data_yaml = data_yaml
        self.imgsz = imgsz
        self.small_target_threshold = small_target_threshold
        self.confusion_classes = confusion_classes
        self.device = device

        self.model = YOLO(model_path)
        self.conv_layers = self._get_conv_layers()

        # Determine class names from the data YAML
        self.class_names = self._load_class_names()
        if confusion_classes:
            for cls in confusion_classes:
                if cls not in self.class_names:
                    print(f"WARNING: class '{cls}' not found in dataset. "
                            f"Available: {self.class_names}")
                    print("Discrimination analysis will be limited.")

        print(f"Found {len(self.conv_layers)} Conv2d layers to analyze")
        print(f"Device: {device}")
        if confusion_classes:
            print(f"Discrimination tracking: {' vs '.join(confusion_classes)}")
        else:
            print("No confusion_classes specified — localization-only analysis")

    def _load_class_names(self) -> List[str]:
        """Load class names from the data YAML."""
        import yaml
        try:
            with open(self.data_yaml) as f:
                data = yaml.safe_load(f)
            names = data.get('names', [])
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names.keys())]
            return list(names)
        except Exception:
            return []

    def _get_class_index(self, class_name: str) -> Optional[int]:
        """Get the integer index for a class name."""
        if class_name in self.class_names:
            return self.class_names.index(class_name)
        return None

    # Extract per-class AP@0.5 from Ultralytics validation results.
    def _extract_per_class_ap(self) -> Dict[str, float]:
        try:
            metrics = self.model.metrics
            if hasattr(metrics.box, 'ap50'):
                ap50 = metrics.box.ap50
                result = {}
                for i, ap in enumerate(ap50):
                    name = self.class_names[i] if i < len(self.class_names) \
                        else f'class_{i}'
                    result[name] = float(ap)
                return result
        except Exception:
            pass
        return {}

    # Compute drone-vs-bird confusion metrics from the confusion matrix. This returns a dict with the raw confusion matrix and the off-diagonal misclassification rates.
    def _compute_confusion_metrics(self) -> Dict:
      if not self.confusion_classes:
          return {}

      cm = None
      candidates = []
      try:
          candidates.append(getattr(self.model, 'metrics', None))
      except Exception:
          pass
      try:
          candidates.append(getattr(self.model, 'validator', None))
      except Exception:
          pass

      for obj in candidates:
          if obj is None:
              continue
          cand = getattr(obj, 'confusion_matrix', None)
          if cand is not None and hasattr(cand, 'matrix'):
              cm = cand
              break

      if cm is None:
          return {}

      matrix = cm.matrix
      if hasattr(matrix, 'detach'):
          matrix = matrix.detach()
      if hasattr(matrix, 'cpu'):
          matrix = matrix.cpu()
      if hasattr(matrix, 'numpy'):
          matrix = matrix.numpy()
      matrix = np.asarray(matrix)

      if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1] or matrix.shape[0] < 2:
          return {}
      nc = matrix.shape[0] - 1  # last row/col is background

      result = {'confusion_matrix': matrix.tolist(), 'num_classes': nc}

      cls_a = self._get_class_index(self.confusion_classes[0])
      cls_b = self._get_class_index(self.confusion_classes[1])

      if cls_a is not None and cls_b is not None:
          total_a = float(matrix[cls_a, :].sum())
          total_b = float(matrix[cls_b, :].sum())
          a_as_b = float(matrix[cls_a, cls_b]) / max(total_a, 1.0)
          b_as_a = float(matrix[cls_b, cls_a]) / max(total_b, 1.0)
          a_miss = float(matrix[cls_a, nc]) / max(total_a, 1.0)
          b_miss = float(matrix[cls_b, nc]) / max(total_b, 1.0)
          fp_a = float(matrix[nc, cls_a])   # background predicted as A
          fp_b = float(matrix[nc, cls_b])   # background predicted as B
          result.update({
              f'{self.confusion_classes[0]}_as_{self.confusion_classes[1]}': a_as_b,
              f'{self.confusion_classes[1]}_as_{self.confusion_classes[0]}': b_as_a,
              f'{self.confusion_classes[0]}_missed': a_miss,
              f'{self.confusion_classes[1]}_missed': b_miss,
              f'false_positives_as_{self.confusion_classes[0]}': int(fp_a),
              f'false_positives_as_{self.confusion_classes[1]}': int(fp_b),
              'cross_confusion': a_as_b + b_as_a,
          })
      return result

    def _get_conv_layers(self) -> List[Tuple[str, nn.Conv2d]]:
        """Extract all Conv2d layers from the YOLOv11 model with their names."""
        layers = []
        pt_model = self.model.model
        for name, module in pt_model.named_modules():
            if isinstance(module, nn.Conv2d):
                layers.append((name, module))
        return layers

    def _get_small_object_ap(self) -> float:
        """
        Extract small-object AP from COCO evaluation if available.

        COCO stats indices: [AP, AP50, AP75, AP_small, AP_medium, AP_large, AR1, AR10]
        Falls back to mAP@0.75 as a proxy if COCO eval is not available.
        """
        try:
            validator = self.model.validator
            if hasattr(validator, 'coco_eval') and validator.coco_eval is not None:
                return float(validator.coco_eval.stats[3])
        except Exception:
            pass
        # Proxy: mAP@0.75 correlates with small-object detection
        try:
            metrics = self.model.metrics
            if hasattr(metrics.box, 'map75'):
                return float(metrics.box.map75)
        except Exception:
            pass
        return 0.0

    # Quantize a single layer's weights and measure AP impact.
    def measure_layer_sensitivity(
        self,
        layer_name: str,
        bits: int = 8,
        per_channel: bool = True,
    ) -> Dict:
        layer = None
        for name, mod in self.conv_layers:
            if name == layer_name:
                layer = mod
                break
        if layer is None:
            raise ValueError(f"Layer {layer_name} not found")

        original_weight = layer.weight.data.clone()
        param_count = layer.weight.numel()

        # Apply fake quantization to this layer only
        layer.weight.data = fake_quantize_tensor(
            original_weight, bits=bits, per_channel=per_channel
        )

        try:
            metrics = self.model.val(
                data=self.data_yaml,
                imgsz=self.imgsz,
                device=self.device,
                verbose=False,
                plots=bool(self.confusion_classes),  # Need plots for confusion matrix
            )
            result = {
                'layer_name': layer_name,
                'bits': bits,
                'map50': float(metrics.box.map50),
                'map': float(metrics.box.map),
                'ap_small': self._get_small_object_ap(),
                'per_class_ap50': self._extract_per_class_ap(),
                'confusion_metrics': self._compute_confusion_metrics(),
                'param_count': param_count,
            }
        except Exception as e:
            print(f"  Error evaluating {layer_name}: {e}")
            result = {
                'layer_name': layer_name,
                'bits': bits,
                'map50': 0.0,
                'map': 0.0,
                'ap_small': 0.0,
                'per_class_ap50': {},
                'confusion_metrics': {},
                'param_count': param_count,
                'error': str(e),
            }
        finally:
            layer.weight.data = original_weight

        return result

    # Run sensitivity analysis across all layers and bit widths.
    def run_full_analysis(
        self,
        bits_list: List[int] = [4, 8],
        per_channel: bool = True,
        output_path: str = 'sensitivity_results.json',
    ) -> Dict:
        # Baseline (FP32) metrics — need plots for confusion matrix
        print("Measuring baseline (FP32) performance...")
        baseline = self.model.val(
            data=self.data_yaml, imgsz=self.imgsz,
            device=self.device, verbose=False,
            plots=bool(self.confusion_classes),
        )
        baseline_map50 = float(baseline.box.map50)
        baseline_map = float(baseline.box.map)
        baseline_ap_small = self._get_small_object_ap()
        baseline_per_class = self._extract_per_class_ap()
        baseline_confusion = self._compute_confusion_metrics()

        print(f"Baseline: mAP@0.5={baseline_map50:.4f}, mAP={baseline_map:.4f}, "
                f"AP_small={baseline_ap_small:.4f}")
        if baseline_per_class:
            print(f"  Per-class AP@0.5: {baseline_per_class}")
        if baseline_confusion:
            print(f"  Cross-confusion: {baseline_confusion.get('cross_confusion', 0):.4f}")

        results = {
            'baseline': {
                'map50': baseline_map50,
                'map': baseline_map,
                'ap_small': baseline_ap_small,
                'per_class_ap50': baseline_per_class,
                'confusion_metrics': baseline_confusion,
            },
            'layers': {},
        }

        total_runs = len(self.conv_layers) * len(bits_list)
        pbar = tqdm(total=total_runs, desc="Sensitivity Analysis")

        for layer_name, layer in self.conv_layers:
            results['layers'][layer_name] = {}
            for bits in bits_list:
                pbar.set_description(f"Analyzing {layer_name[-30:]} @ {bits}bit")
                metrics = self.measure_layer_sensitivity(
                    layer_name, bits=bits, per_channel=per_channel
                )
                metrics['sensitivity_map50'] = baseline_map50 - metrics['map50']
                metrics['sensitivity_map'] = baseline_map - metrics['map']
                metrics['sensitivity_small'] = baseline_ap_small - metrics['ap_small']

                # Discrimination sensitivity: increase in cross-confusion
                baseline_cross = baseline_confusion.get('cross_confusion', 0)
                quantized_cross = metrics.get('confusion_metrics', {}).get('cross_confusion', 0)
                metrics['sensitivity_discrimination'] = quantized_cross - baseline_cross

                # Per-class AP drop
                if baseline_per_class and metrics.get('per_class_ap50'):
                    per_class_drop = {}
                    for cls_name, baseline_ap in baseline_per_class.items():
                        quantized_ap = metrics['per_class_ap50'].get(cls_name, 0)
                        per_class_drop[cls_name] = baseline_ap - quantized_ap
                    metrics['sensitivity_per_class'] = per_class_drop

                results['layers'][layer_name][str(bits)] = metrics
                pbar.update(1)

        pbar.close()

        with open(output_path, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\nResults saved to {output_path}")

        self._print_summary(results, bits_list)
        return results

    @staticmethod
    def _print_summary(results: Dict, bits_list: List[int]):
        print("\n" + "=" * 70)
        print("SENSITIVITY ANALYSIS SUMMARY")
        print("=" * 70)
        for bits in bits_list:
            print(f"\n{bits}-bit quantization:")

            # Localization sensitivity (small-target AP drop)
            loc_sens = []
            for name, layer_results in results['layers'].items():
                if str(bits) in layer_results:
                    s = layer_results[str(bits)].get('sensitivity_small', 0)
                    loc_sens.append((name, s))
            loc_sens.sort(key=lambda x: -x[1])
            print(f"  Top 5 localization-sensitive layers (small-target AP drop):")
            for name, s in loc_sens[:5]:
                print(f"    {name}: {s:.4f}")
            print(f"  Top 5 least localization-sensitive layers:")
            for name, s in loc_sens[-5:]:
                print(f"    {name}: {s:.4f}")

            # Discrimination sensitivity (cross-confusion increase)
            disc_sens = []
            for name, layer_results in results['layers'].items():
                if str(bits) in layer_results:
                    s = layer_results[str(bits)].get('sensitivity_discrimination', 0)
                    disc_sens.append((name, s))
            if disc_sens and any(s != 0 for _, s in disc_sens):
                disc_sens.sort(key=lambda x: -x[1])
                print(f"  Top 5 discrimination-sensitive layers (confusion increase):")
                for name, s in disc_sens[:5]:
                    print(f"    {name}: {s:.4f}")
                print(f"  Top 5 least discrimination-sensitive layers:")
                for name, s in disc_sens[-5:]:
                    print(f"    {name}: {s:.4f}")

            # Show layers where localization and discrimination sensitivity diverge
            if disc_sens and any(s != 0 for _, s in disc_sens):
                loc_rank = {name: i for i, (name, _) in enumerate(loc_sens)}
                disc_rank = {name: i for i, (name, _) in enumerate(disc_sens)}
                divergences = []
                for name in loc_rank:
                    if name in disc_rank:
                        divergences.append((name, abs(loc_rank[name] - disc_rank[name])))
                divergences.sort(key=lambda x: -x[1])
                print(f"  Top 5 divergence layers (localization vs discrimination rank mismatch):")
                for name, d in divergences[:5]:
                    print(f"    {name}: rank diff={d} "
                            f"(loc_rank={loc_rank[name]+1}, disc_rank={disc_rank[name]+1})")

### **Section 3: Mixed-Precision Bit Allocation**

This allocates per-layer bit widths based on sensitivity scores.

Given the sensitivity analysis results, determines the optimal bit width
for each layer such that:
- Total model size stays within a budget (e.g., 35% of FP32)
- Total detection accuracy (especially small-target AP) is maximized
- Drone-vs-bird discrimination is preserved (when discrimination data exists)

Two algorithms:
1. Greedy: Start all at lowest precision, upgrade most sensitive layers first
2. Dynamic Programming: Optimal allocation via knapsack formulation

The DP approach guarantees the global optimum under the discretized budget,
while the greedy approach is faster and produces near-optimal results.

Objective:
The combined sensitivity score for each layer is:
    $$combined = (w_{loc} \times sensitivity_{small}) + (w_{disc} \times sensitivity_{discrimination})$$

When discrimination data is not available (no confusion_classes in the sensitivity analysis), w_disc is set to 0 and the allocator falls back to localization-only optimization (backward-compatible).

In [15]:
class MixedPrecisionAllocator:
    def __init__(
            self,
            sensitivity_results: Dict,
            target_size_ratio: float = 0.35,
            available_bits: List[int] = [4, 8, 16],
            w_loc: float = 0.6,
            w_disc: float = 0.4,
        ):
            self.results = sensitivity_results
            self.target_ratio = target_size_ratio
            self.available_bits = sorted(available_bits)
            self.w_loc = w_loc
            self.w_disc = w_disc

            # Check if discrimination data is available
            has_disc = False
            for layer_data in sensitivity_results['layers'].values():
                for bits_data in layer_data.values():
                    if 'sensitivity_discrimination' in bits_data:
                        has_disc = True
                        break
                if has_disc:
                    break

            if not has_disc:
                self.w_disc = 0.0
                if w_disc > 0:
                    print("WARNING: No discrimination sensitivity data found. "
                        "Falling back to localization-only optimization.")

            self.layers = []
            for layer_name, bit_results in sensitivity_results['layers'].items():
                param_count = list(bit_results.values())[0]['param_count']
                loc_sens = {}
                disc_sens = {}
                for bits_str, metrics in bit_results.items():
                    loc_sens[int(bits_str)] = metrics.get('sensitivity_small', 0)
                    disc_sens[int(bits_str)] = metrics.get('sensitivity_discrimination', 0)

                combined = {}
                for bits in loc_sens:
                    combined[bits] = (
                        self.w_loc * loc_sens[bits] +
                        self.w_disc * disc_sens.get(bits, 0)
                    )

                self.layers.append({
                    'name': layer_name,
                    'param_count': param_count,
                    'sensitivities': combined,
                    'loc_sensitivities': loc_sens,
                    'disc_sensitivities': disc_sens,
                })

            total_params = sum(l['param_count'] for l in self.layers)
            self.fp32_size_bytes = total_params * 4
            self.target_size_bytes = self.fp32_size_bytes * target_size_ratio

            print(f"Total layers: {len(self.layers)}")
            print(f"FP32 model size: {self.fp32_size_bytes / 1024 / 1024:.2f} MB")
            print(f"Target size ({target_size_ratio*100:.0f}%): "
                f"{self.target_size_bytes / 1024 / 1024:.2f} MB")
            print(f"Objective weights: w_loc={self.w_loc}, w_disc={self.w_disc}")

    def _layer_size_bytes(self, param_count: int, bits: int) -> float:
        return param_count * (bits / 8)

    def greedy_allocate(self) -> Dict[str, int]:
        """
        Greedy allocation: start all layers at lowest precision, then upgrade
        the most sensitive layers (per-byte sensitivity reduction) until the
        size budget is exhausted.

        Returns: Dict mapping layer_name -> bits
        """
        min_bits = min(self.available_bits)
        allocation = {l['name']: min_bits for l in self.layers}

        current_size = sum(
            self._layer_size_bytes(l['param_count'], min_bits) for l in self.layers
        )

        print(f"\nGreedy allocation:")
        print(f"  Initial size (all {min_bits}-bit): "
            f"{current_size / 1024 / 1024:.2f} MB")

        improved = True
        iterations = 0
        while improved and iterations < 2000:
            improved = False
            iterations += 1

            best_ratio = -1
            best_layer = None
            best_new_bits = None

            for layer in self.layers:
                current_bits = allocation[layer['name']]
                idx = self.available_bits.index(current_bits)
                if idx >= len(self.available_bits) - 1:
                    continue

                new_bits = self.available_bits[idx + 1]
                size_increase = (
                    self._layer_size_bytes(layer['param_count'], new_bits) -
                    self._layer_size_bytes(layer['param_count'], current_bits)
                )
                if size_increase <= 0:
                    continue

                # Sensitivity reduction (benefit) per byte (cost)
                current_sens = layer['sensitivities'].get(current_bits, 0)
                new_sens = layer['sensitivities'].get(new_bits, 0)
                sens_gain = current_sens - new_sens
                ratio = sens_gain / size_increase

                if ratio > best_ratio:
                    new_size = current_size + size_increase
                    if new_size <= self.target_size_bytes:
                        best_ratio = ratio
                        best_layer = layer
                        best_new_bits = new_bits

            if best_layer is not None:
                old_bits = allocation[best_layer['name']]
                allocation[best_layer['name']] = best_new_bits
                current_size += (
                    self._layer_size_bytes(best_layer['param_count'], best_new_bits) -
                    self._layer_size_bytes(best_layer['param_count'], old_bits)
                )
                improved = True

        self._print_allocation_stats(allocation, "Greedy")
        return allocation

    def dp_allocate(self, granularity: int = 200) -> Dict[str, int]:
        """
        Dynamic programming allocation (optimal knapsack).

        Discretizes the size budget into 'granularity' steps and solves
        a 0/1 multiple-choice knapsack to minimize total sensitivity.

        Args:
            granularity: Number of budget steps (higher = more precise)

        Returns: Dict mapping layer_name -> bits
        """
        n = len(self.layers)
        budget = int(self.target_size_bytes)
        step = max(1, budget // granularity)
        budget_steps = budget // step

        # Precompute (bits, size_in_steps, sensitivity) for each layer
        options = []
        for layer in self.layers:
            layer_opts = []
            for bits in self.available_bits:
                size = int(self._layer_size_bytes(
                    layer['param_count'], bits) / step)
                sens = layer['sensitivities'].get(bits, 0)
                layer_opts.append((bits, size, sens))
            options.append(layer_opts)

        # DP table: dp[i][j] = min sensitivity using first i layers with j steps
        dp = [[float('inf')] * (budget_steps + 1) for _ in range(n + 1)]
        choice = [[None] * (budget_steps + 1) for _ in range(n + 1)]
        dp[0][0] = 0

        for i in range(1, n + 1):
            for j in range(budget_steps + 1):
                for bits, size, sens in options[i - 1]:
                    if j >= size and dp[i - 1][j - size] + sens < dp[i][j]:
                        dp[i][j] = dp[i - 1][j - size] + sens
                        choice[i][j] = bits

        best_j = min(range(budget_steps + 1), key=lambda j: dp[n][j])

        if dp[n][best_j] == float('inf'):
            print("DP failed to find feasible solution, falling back to greedy")
            return self.greedy_allocate()

        # Backtrack
        allocation = {}
        j = best_j
        for i in range(n, 0, -1):
            bits = choice[i][j]
            allocation[self.layers[i - 1]['name']] = bits
            size = int(self._layer_size_bytes(
                self.layers[i - 1]['param_count'], bits) / step)
            j -= size

        self._print_allocation_stats(allocation, "DP")
        return allocation

    def _print_allocation_stats(self, allocation: Dict[str, int], method: str):
        final_size = sum(
            self._layer_size_bytes(l['param_count'], allocation[l['name']])
            for l in self.layers
        )
        print(f"\n{method} allocation:")
        print(f"  Final size: {final_size / 1024 / 1024:.2f} MB "
            f"({final_size / self.fp32_size_bytes * 100:.1f}% of FP32)")
        for bits in self.available_bits:
            count = sum(1 for b in allocation.values() if b == bits)
            print(f"  {bits}-bit layers: {count}")

        # Report objective breakdown for the allocation
        total_loc = 0.0
        total_disc = 0.0
        for layer in self.layers:
            bits = allocation[layer['name']]
            total_loc += layer['loc_sensitivities'].get(bits, 0)
            total_disc += layer['disc_sensitivities'].get(bits, 0)
        print(f"  Total localization sensitivity: {total_loc:.4f}")
        if self.w_disc > 0:
            print(f"  Total discrimination sensitivity: {total_disc:.4f}")
            print(f"  Combined objective: "
                f"{self.w_loc * total_loc + self.w_disc * total_disc:.4f}")

    def save_allocation(self, allocation: Dict[str, int], path: str):
        with open(path, 'w') as f:
            json.dump(allocation, f, indent=2)
        print(f"Allocation saved to {path}")

    @staticmethod
    def load_allocation(path: str) -> Dict[str, int]:
        with open(path) as f:
            return {k: int(v) for k, v in json.load(f).items()}

### **Section 4: QAT with Small-Target Weighted Loss**

This is a wrapper for YOLOv11 detection loss with area-based target weighting and inter-class confusion regularization.

The standard YOLOv11 loss treats all targets equally. This modified loss up-weights the localization (CIoU) and distribution focal loss (DFL)for small bounding boxes, encouraging the model to better detect small, distant drones that are most affected by quantization.

Weight formula: $$w = clamp\left(\frac{area_{scale}}{target_{area}}, min_{weight}, max_{weight}\right)$$

A target with area 16x16=256 gets ~4x the weight of a 32x32=1024 target.
This is critical because quantization disproportionately degrades the
high-frequency features needed to localize small targets.

Additionally, when confusion_classes are provided, a margin-based
confusion regularization term is added to the loss. This term penalizes
the model when the classification logits for drone and bird targets are
too close, i.e., when the decision boundary between these two visually
similar classes is poorly separated. This is especially important under
quantization, which compresses the logit space and makes fine-grained
discrimination harder.

In [16]:
class SmallTargetWeightedLoss(nn.Module):
    def __init__(
        self,
        area_scale: float = 32 * 32,
        min_weight: float = 0.5,
        max_weight: float = 4.0,
        confusion_classes: Optional[List[str]] = None,
        class_names: Optional[List[str]] = None,
        confusion_margin: float = 2.0,
        confusion_weight: float = 0.1,
    ):
        super().__init__()
        self.area_scale = area_scale
        self.min_weight = min_weight
        self.max_weight = max_weight
        self.confusion_classes = confusion_classes
        self.class_names = class_names or []
        self.confusion_margin = confusion_margin
        self.confusion_weight = confusion_weight

        # Determine class indices for confusion regularization
        self.cls_a_idx = None
        self.cls_b_idx = None
        if confusion_classes and class_names:
            if confusion_classes[0] in class_names:
                self.cls_a_idx = class_names.index(confusion_classes[0])
            if confusion_classes[1] in class_names:
                self.cls_b_idx = class_names.index(confusion_classes[1])

    # Compute per-target weights based on bounding box area.
    def compute_area_weights(self, target_bboxes: torch.Tensor) -> torch.Tensor:
        widths = target_bboxes[:, 2] - target_bboxes[:, 0]
        heights = target_bboxes[:, 3] - target_bboxes[:, 1]
        areas = (widths * heights).clamp(min=1.0)
        weights = self.area_scale / areas
        return weights.clamp(self.min_weight, self.max_weight)

    # Margin-based regularization to separate drone and bird logits.
    def compute_confusion_regularization(
        self,
        cls_logits: torch.Tensor,
        target_classes: torch.Tensor,
    ) -> torch.Tensor:
        if self.cls_a_idx is None or self.cls_b_idx is None:
            return torch.tensor(0.0, device=cls_logits.device)

        # Find targets of class A and class B
        mask_a = (target_classes == self.cls_a_idx)
        mask_b = (target_classes == self.cls_b_idx)

        loss = torch.tensor(0.0, device=cls_logits.device)

        if mask_a.any():
            # For drone targets: logit[drone] - logit[bird] should be > margin
            logits_a = cls_logits[mask_a]
            diff_a = logits_a[:, self.cls_a_idx] - logits_a[:, self.cls_b_idx]
            # Hinge loss: max(0, margin - diff)
            loss = loss + torch.relu(self.confusion_margin - diff_a).mean()

        if mask_b.any():
            # For bird targets: logit[bird] - logit[drone] should be > margin
            logits_b = cls_logits[mask_b]
            diff_b = logits_b[:, self.cls_b_idx] - logits_b[:, self.cls_a_idx]
            loss = loss + torch.relu(self.confusion_margin - diff_b).mean()

        return self.confusion_weight * loss

# Insert FakeQuantize hooks into the YOLOv11 model for QAT.
def prepare_model_for_qat(
    model: nn.Module,
    bit_allocation: Optional[Dict[str, int]] = None,
    default_bits: int = 8
) -> nn.Module:

    quantized_count = 0
    full_precision_count = 0

    for name, module in model.named_modules():
        if not isinstance(module, nn.Conv2d):
            continue

        bits = default_bits
        if bit_allocation and name in bit_allocation:
            bits = bit_allocation[name]

        if bits >= 16:
            full_precision_count += 1
            continue

        # Create a closure capturing the bit width for this layer
        def make_quantize_hook(num_bits: int):
            qmax = 2 ** (num_bits - 1) - 1

            def hook(mod, inputs):
                with torch.no_grad():
                    # Per-channel symmetric quantization on weights
                    weight = mod.weight.data
                    max_per_channel = weight.reshape(weight.shape[0], -1).abs().amax(dim=1)
                    scales = (max_per_channel / qmax).clamp(min=1e-8)
                    scales = scales.view(-1, *([1] * (weight.dim() - 1)))
                    q = torch.round(weight / scales).clamp(-qmax, qmax)
                    mod.weight.data = q * scales

            return hook

        module.register_forward_pre_hook(make_quantize_hook(bits))
        quantized_count += 1

    print(f"QAT preparation: {quantized_count} layers quantized, "
          f"{full_precision_count} layers at full precision")
    return model

**QAT Training**

This trainer:
1. Inserts FakeQuantize hooks into the model
2. Patches the YOLOv11 loss to up-weight small-target localization
3. Trains for `N` epochs with quantization-aware gradients
4. Exports the final quantized model for edge deployment

The training process teaches the model to maintain accuracy under integer quantization, specifically preserving small-target detection.

The small-target loss modification works by injecting area-based weights into the box loss computation. Targets with smaller bounding boxes (distant drones) receive higher loss weights, causing the optimizer to prioritize accurate localization of these hard cases.

In [28]:
import numpy as np
from pathlib import Path
import hashlib
from typing import Dict, List, Optional, Tuple

# Custom QAT training loop for YOLOv11 with small-target-weighted loss.
class QATTrainer:
    def __init__(
        self,
        model_path: str,
        data_yaml: str,
        bit_allocation: Optional[Dict[str, int]] = None,
        epochs: int = 50,
        lr: float = 0.01,
        imgsz: int = 640,
        batch: int = 16,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        area_scale: float = 32 * 32,
    ):
        from ultralytics import YOLO

        self.model = YOLO(model_path)
        self.data_yaml = data_yaml
        self.bit_allocation = bit_allocation
        self.epochs = epochs
        self.lr = lr
        self.imgsz = imgsz
        self.batch = batch
        self.device = device
        self.area_scale = area_scale

        # Prepare model for QAT (inserts fake quantization hooks)
        pt_model = self.model.model
        prepare_model_for_qat(pt_model, bit_allocation)

        print(f"QAT Trainer initialized:")
        print(f"  Epochs: {epochs}")
        print(f"  Learning rate: {lr}")
        print(f"  Mixed precision: {'Yes' if bit_allocation else 'No (uniform INT8)'}")

    def train(self) -> str:
        import ultralytics.utils.loss as loss_module

        # Save original loss class
        original_loss = loss_module.v8DetectionLoss

        # Create a patched loss class with small-target weighting
        class PatchedLoss(original_loss):
            def __init__(self, model, area_scale=32 * 32,
                         min_weight=0.5, max_weight=4.0,
                         confusion_classes=None, class_names=None,
                         confusion_margin=2.0, confusion_weight=0.1,
                         imgsz=640):
                super().__init__(model)
                self.area_scale = area_scale
                self.min_weight = min_weight
                self.max_weight = max_weight
                self.confusion_classes = confusion_classes
                self.class_names = class_names or []
                self.confusion_margin = confusion_margin
                self.confusion_weight = confusion_weight
                self.imgsz = imgsz

                # Determine class indices for confusion regularization
                self.cls_a_idx = None
                self.cls_b_idx = None
                if confusion_classes and class_names:
                    if confusion_classes[0] in class_names:
                        self.cls_a_idx = class_names.index(confusion_classes[0])
                    if confusion_classes[1] in class_names:
                        self.cls_b_idx = class_names.index(confusion_classes[1])

            def __call__(self, preds, batch):
                # Compute area weights for targets and inject into loss
                if 'bboxes' in batch and len(batch['bboxes']) > 0:
                    bboxes = batch['bboxes']  # (N, 4) in xywh normalized
                    areas = bboxes[:, 2] * bboxes[:, 3]  # w * h (normalized)
                    # Denormalize area to pixel space
                    pixel_areas = areas * (self.imgsz ** 2)
                    weights = (self.area_scale / (pixel_areas + 1e-6))
                    weights = weights.clamp(self.min_weight, self.max_weight)
                    self._area_weights = weights

                # Compute base loss
                total_loss = super().__call__(preds, batch)

                # Add confusion regularization if enabled
                if self.cls_a_idx is not None and self.cls_b_idx is not None:
                    reg_loss = self._compute_confusion_reg(preds, batch)
                    if isinstance(total_loss, (tuple, list)):
                        total_loss = (total_loss[0] + reg_loss,) + total_loss[1:]
                    else:
                        total_loss = total_loss + reg_loss

                return total_loss

            def _compute_confusion_reg(self, preds, batch) -> torch.Tensor:
                if 'cls' not in batch:
                    return torch.tensor(0.0, device=preds[0].device
                                        if isinstance(preds, (list, tuple))
                                        else preds.device)

                cls_labels = batch.get('cls', None)
                if cls_labels is None:
                    return torch.tensor(0.0)

                try:
                    if isinstance(preds, (list, tuple)):
                        pred = preds[0]
                    else:
                        pred = preds

                    nc = len(self.class_names) if self.class_names else None
                    if nc is None or pred.shape[-1] < nc + 4:
                        return torch.tensor(0.0, device=pred.device)

                    # pred shape: (B, A, nc+4) for YOLOv11 classification head
                    cls_logits = pred[..., 4:4+nc]  # (B, A, nc)
                    mean_logits = cls_logits.mean(dim=1)  # (B, nc)

                    batch_labels = batch.get('batch_idx', None)
                    if batch_labels is None:
                        return torch.tensor(0.0, device=pred.device)

                    loss = torch.tensor(0.0, device=pred.device)
                    num_images = int(batch_labels.max().item() + 1) \
                        if len(batch_labels) > 0 else 0

                    for img_idx in range(num_images):
                        mask = (batch_labels == img_idx)
                        if not mask.any():
                            continue
                        img_classes = cls_labels[mask]
                        if img_classes.dim() > 1:
                            img_classes = img_classes.squeeze(-1)

                        has_a = (img_classes == self.cls_a_idx).any()
                        has_b = (img_classes == self.cls_b_idx).any()

                        if has_a:
                            diff = (mean_logits[img_idx, self.cls_a_idx] -
                                    mean_logits[img_idx, self.cls_b_idx])
                            loss = loss + torch.relu(self.confusion_margin - diff)
                        if has_b:
                            diff = (mean_logits[img_idx, self.cls_b_idx] -
                                    mean_logits[img_idx, self.cls_a_idx])
                            loss = loss + torch.relu(self.confusion_margin - diff)

                    return self.confusion_weight * loss / max(num_images, 1)
                except Exception:
                    return torch.tensor(0.0)

        # Replace the loss class globally
        loss_module.v8DetectionLoss = lambda model, **kwargs: PatchedLoss(
            model,
            area_scale=self.area_scale,
            confusion_classes=self.confusion_classes,
            class_names=self.class_names,
            confusion_margin=self.confusion_margin,
            confusion_weight=self.confusion_weight,
            imgsz=self.imgsz,
        )

        try:
            results = self.model.train(
                data=self.data_yaml,
                epochs=self.epochs,
                lr0=self.lr,
                imgsz=self.imgsz,
                batch=self.batch,
                device=self.device,
                warmup_epochs=3,
                cos_lr=True,
                save=True,
                project='runs/qat',
                name='drone_qat',
            )
        finally:
            # Always restore the original loss class
            loss_module.v8DetectionLoss = original_loss

        best_path = str(Path('/content/runs/detect/runs/qat/drone_qat/weights/best.pt'))
        print(f"\nQAT training complete. Best model: {best_path}")
        return best_path

    def _fake_quant_per_channel(self, weight_np: np.ndarray, bits: int):
        """
        Symmetric per-channel fake quantization. MUST match the QAT
        forward_pre_hook scheme in prepare_model_for_qat() exactly:
            qmax  = 2^(bits-1) - 1              (127 for 8-bit, 7 for 4-bit)
            scale = per-channel |w|max / qmax   (clamped at 1e-8)
            q     = round(w / scale).clip(-qmax, qmax)
        Returns (q_int8, scale_f32[C], dequantized_f32).
        """
        qmax = 2 ** (bits - 1) - 1
        w = weight_np.astype(np.float64)
        flat = w.reshape(w.shape[0], -1)
        max_per_ch = np.abs(flat).max(axis=1)
        scale = np.maximum(max_per_ch / qmax, 1e-8).astype(np.float32)
        scale_b = scale.reshape(-1, *([1] * (w.ndim - 1))).astype(np.float64)
        q = np.clip(np.round(w / scale_b), -qmax, qmax)
        deq = (q * scale_b).astype(np.float32)
        return q.astype(np.int8), scale, deq


    def _build_value_index(self, reference_weights: Dict[str, np.ndarray]) -> Dict[str, List[str]]:
        """Index reference weights by content hash -> [layer names]."""
        index: Dict[str, List[str]] = {}
        for name, w in reference_weights.items():
            w32 = np.ascontiguousarray(np.asarray(w, dtype=np.float32))
            key = hashlib.md5(w32.tobytes()).hexdigest() + f"|{w32.shape}"
            index.setdefault(key, []).append(name)
        return index


    def export_mixed_precision_onnx(
        self,
        fp32_onnx_path: str,
        reference_weights: Dict[str, np.ndarray],
        bit_allocation: Dict[str, int],
        output_path: str,
        default_bits: int = 8,
    ) -> Dict:
        """
        Bake the per-layer mixed-precision scheme into an ONNX graph.

        For every Conv node whose layer is assigned bits < 16:
            - the FLOAT32 weight initializer is replaced by an INT8 initializer
              holding the quantized levels (4-bit levels stored one-per-byte)
            - a per-channel FLOAT32 scale initializer is added
            - a DequantizeLinear(axis=0) node is inserted before the Conv
        Layers assigned 16 bits keep their FLOAT32 weights untouched.

        Args:
            fp32_onnx_path: FP32 ONNX export (YOLO.export(format='onnx', opset=13)).
            reference_weights: {pytorch_layer_name: conv weight ndarray}, from the
                SAME checkpoint that produced the ONNX export:
                {n: m.weight.data.cpu().numpy().copy()
                for n, m in model.model.named_modules() if isinstance(m, nn.Conv2d)}
            bit_allocation: {layer_name: bits} from Stage 2.
            output_path: where to save the mixed-precision model.
            default_bits: bits for ONNX convs missing from the allocation.

        Returns: report dict (per-bit counts, unmatched layers, sizes).
        Requires opset >= 13 (DequantizeLinear axis attribute).
        """
        import onnx
        from onnx import helper, numpy_helper

        model = onnx.load(fp32_onnx_path)
        graph = model.graph

        opset = next((o.version for o in model.opset_import
                      if o.domain in ('', 'ai.onnx')), None)
        if opset is not None and opset < 13:
            raise RuntimeError(f"ONNX opset {opset} < 13; DequantizeLinear(axis=) "
                              f"needs opset >= 13. Re-export with opset=13.")

        inits = {i.name: i for i in graph.initializer}
        ref_index = self._build_value_index(reference_weights)
        used_names = set()

        def match_layer(w: np.ndarray) -> Optional[str]:
            w32 = np.ascontiguousarray(np.asarray(w, dtype=np.float32))
            key = hashlib.md5(w32.tobytes()).hexdigest() + f"|{w32.shape}"
            names = ref_index.get(key)
            if not names:
                return None
            for n in names:
                if n not in used_names:
                    used_names.add(n)
                    return n
            return None

        # pass 1: map Conv nodes -> layer names
        plan = {}
        matched, unmatched_convs, skipped = [], [], []
        for idx, node in enumerate(graph.node):
            if node.op_type != 'Conv' or len(node.input) < 2:
                continue
            wname = node.input[1]
            winit = inits.get(wname)
            if winit is None:
                skipped.append(wname)
                continue
            layer = match_layer(numpy_helper.to_array(winit))
            if layer is None:
                unmatched_convs.append(wname)
                continue
            plan[idx] = (layer, int(bit_allocation.get(layer, default_bits)), wname)
            matched.append(layer)

        unmatched_alloc = [n for n in bit_allocation if n not in matched]

        # pass 2: surgery
        new_nodes = []
        stats = {'16': 0, '8': 0, '4': 0}
        for idx, node in enumerate(graph.node):
            if idx not in plan:
                new_nodes.append(node)
                continue
            layer, bits, wname = plan[idx]
            winit = inits[wname]
            w = numpy_helper.to_array(winit)

            if bits >= 16:
                stats['16'] += 1
                new_nodes.append(node)
                continue

            q_int8, scale, _ = self._fake_quant_per_channel(w, bits)
            stats[str(bits)] = stats.get(str(bits), 0) + 1

            base = f"mpq_{layer}_w{bits}".replace('.', '_')
            q_name, s_name, dq_name = f"{base}_q", f"{base}_scale", f"{base}_dq"

            graph.initializer.remove(winit)
            graph.initializer.append(numpy_helper.from_array(q_int8, q_name))
            graph.initializer.append(numpy_helper.from_array(scale, s_name))

            dq_node = helper.make_node(
                'DequantizeLinear', [q_name, s_name], [dq_name],
                axis=0, name=f"{base}_DequantizeLinear")
            new_nodes.append(dq_node)

            conv = onnx.NodeProto()
            conv.CopyFrom(node)
            conv.input[1] = dq_name
            new_nodes.append(conv)

        del graph.node[:]
        graph.node.extend(new_nodes)

        try:
            onnx.checker.check_model(model)
        except Exception as e:
            print(f"WARNING: onnx.checker flagged the model ({e}); saving anyway.")

        onnx.save(model, output_path)

        import os
        fp32_bytes = os.path.getsize(fp32_onnx_path)
        mp_bytes = os.path.getsize(output_path)
        report = {
            'output_path': output_path,
            'layers_16bit': stats['16'],
            'layers_8bit': stats.get('8', 0),
            'layers_4bit': stats.get('4', 0),
            'matched_layers': len(matched),
            'unmatched_onnx_convs': unmatched_convs,
            'allocation_layers_not_found': unmatched_alloc,
            'skipped_non_initializer_weights': skipped,
            'fp32_onnx_bytes': fp32_bytes,
            'mixed_precision_onnx_bytes': mp_bytes,
            'size_ratio': round(mp_bytes / max(fp32_bytes, 1), 4),
        }
        print("Mixed-precision ONNX export complete:")
        print(f"  16-bit (FP32 kept): {report['layers_16bit']} layers")
        print(f"  8-bit (INT8 QDQ):   {report['layers_8bit']} layers")
        print(f"  4-bit (INT4 in INT8 storage): {report['layers_4bit']} layers")
        print(f"  Size: {fp32_bytes/1e6:.2f} MB -> {mp_bytes/1e6:.2f} MB "
              f"({report['size_ratio']*100:.1f}%)")
        if unmatched_convs or unmatched_alloc:
            print(f"  WARNING unmatched ONNX convs: {unmatched_convs[:5]}")
            print(f"  WARNING allocation layers not found: {unmatched_alloc[:5]}")
        return report

    def export_quantized(
        self,
        model_path: str,
        output_format: str = 'onnx',
        output_path: str = 'drone_quantized.onnx',
    ) -> str:
        """
        Export the QAT-trained model for edge deployment.

        CHANGED: when self.bit_allocation exists (mixed-precision run), the ONNX
        export bakes the per-layer precision into the graph via
        export_mixed_precision_onnx() instead of uniform dynamic INT8.
        Uniform quantize_dynamic remains only as the no-allocation fallback.
        """
        from ultralytics import YOLO

        model = YOLO(model_path)

        if output_format == 'onnx':
            fp32_path = model.export(
                format='onnx', imgsz=self.imgsz, opset=13,
                simplify=True, dynamic=False,
            )
            if self.bit_allocation:
                import torch.nn as nn
                ref = {n: m.weight.data.cpu().numpy().copy()
                      for n, m in model.model.named_modules()
                      if isinstance(m, nn.Conv2d)}
                self.export_mixed_precision_onnx(
                    fp32_path, ref, self.bit_allocation, output_path,
                    default_bits=8,
                )
            else:
                try:
                    from onnxruntime.quantization import quantize_dynamic, QuantType
                    print("No bit_allocation — uniform INT8 dynamic quantization.")
                    quantize_dynamic(fp32_path, output_path, weight_type=QuantType.QUInt8)
                    print(f"Quantized ONNX model saved to: {output_path}")
                except ImportError:
                    print("onnxruntime not installed. Exporting FP32 ONNX instead.")
                    output_path = fp32_path

        elif output_format == 'ncnn':
            output_path = model.export(format='ncnn', imgsz=self.imgsz)
            print(f"NCNN model saved to: {output_path}")

        elif output_format == 'openvino':
            output_path = model.export(format='openvino', imgsz=self.imgsz)
            print(f"OpenVINO model saved to: {output_path}")

        elif output_format == 'tflite':
            output_path = model.export(format='tflite', imgsz=self.imgsz, int8=True)
            print(f"TFLite INT8 model saved to: {output_path}")

        else:
            raise ValueError(f"Unsupported format: {output_format}")

        return output_path

### **Section 5: Motion-Gated Inference**

Two-stage detection pipeline: temporal motion gating + quantized YOLOv8n.

1. Stage 1 (Near-zero cost): Frame differencing or MOG2 background subtraction identifies regions with movement. If no motion is detected, skip detection entirely, saving full inference cost.

2. Stage 2 (Quantized YOLO11n): Run detection only on motion candidate regions (crops) rather than the full frame. This increases effective resolution per target and reduces total compute.

On CPU-only edge hardware, this reduces average inference time by 60-80% on sky-dominated surveillance video while improving small-target recall (higher effective resolution per crop).

Safety mechanism: A full-frame detection fallback runs every `N` frames to catch slow-moving or hovering drones that frame differencing misses.

**Parameters:**
- `motion_threshold`: Pixel difference threshold for motion detection
- `min_area`: Minimum contour area to be considered a candidate (pixels)
- `motion_ratio_threshold`: Skip detection if motion covers < this fraction
- `fallback_interval`: Run full-frame detection every N frames as safety net
- `use_mog2`: Use MOG2 background subtractor (more robust than frame diff)

In [18]:
import numpy as np
import cv2
from typing import List, Tuple, Dict, Optional
import time

class MotionGatedDetector:

    def __init__(
        self,
        model_path: str,
        motion_threshold: int = 25,
        min_area: int = 100,
        dilation_kernel: Tuple[int, int] = (15, 15),
        motion_ratio_threshold: float = 0.005,
        fallback_interval: int = 30,
        use_mog2: bool = True,
        mog2_history: int = 500,
        mog2_var_threshold: int = 16,
        imgsz: int = 640,
        conf_threshold: float = 0.25,
        iou_threshold: float = 0.45,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
    ):
        from ultralytics import YOLO

        self.model = YOLO(model_path)
        self.motion_threshold = motion_threshold
        self.min_area = min_area
        self.dilation_kernel = np.ones(dilation_kernel, np.uint8)
        self.motion_ratio_threshold = motion_ratio_threshold
        self.fallback_interval = fallback_interval
        self.imgsz = imgsz
        self.conf_threshold = conf_threshold
        self.iou_threshold = iou_threshold
        self.device = device

        if use_mog2:
            self.bg_subtractor = cv2.createBackgroundSubtractorMOG2(
                history=mog2_history,
                varThreshold=mog2_var_threshold,
                detectShadows=False,
            )
        else:
            self.bg_subtractor = None

    def detect_motion_regions(
        self,
        prev_frame: np.ndarray,
        curr_frame: np.ndarray,
    ) -> Tuple[List[Tuple[int, int, int, int]], float]:
        h, w = curr_frame.shape[:2]

        if self.bg_subtractor is not None:
            mask = self.bg_subtractor.apply(curr_frame)
        else:
            prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
            curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
            diff = cv2.absdiff(prev_gray, curr_gray)
            _, mask = cv2.threshold(diff, self.motion_threshold, 255, cv2.THRESH_BINARY)

        # Clean up mask with morphology
        mask = cv2.dilate(mask, self.dilation_kernel, iterations=2)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, self.dilation_kernel)

        # Find contours and create bounding boxes
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        boxes = []
        for contour in contours:
            if cv2.contourArea(contour) > self.min_area:
                x, y, bw, bh = cv2.boundingRect(contour)
                # Pad box for context (YOLO needs surrounding context)
                pad = max(bw, bh) // 2
                boxes.append((
                    max(0, x - pad), max(0, y - pad),
                    min(w, x + bw + pad), min(h, y + bh + pad),
                ))

        boxes = self._merge_boxes(boxes)

        motion_ratio = cv2.countNonZero(mask) / (h * w)
        return boxes, motion_ratio

    @staticmethod
    def _merge_boxes(
        boxes: List[Tuple[int, int, int, int]],
        overlap_threshold: float = 0.3,
    ) -> List[Tuple[int, int, int, int]]:
        """Merge overlapping bounding boxes."""
        if not boxes:
            return []

        boxes = sorted(boxes, key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)
        merged = []
        used = [False] * len(boxes)

        for i in range(len(boxes)):
            if used[i]:
                continue
            x1, y1, x2, y2 = boxes[i]
            for j in range(i + 1, len(boxes)):
                if used[j]:
                    continue
                ix1, iy1 = max(x1, boxes[j][0]), max(y1, boxes[j][1])
                ix2, iy2 = min(x2, boxes[j][2]), min(y2, boxes[j][3])
                if ix1 < ix2 and iy1 < iy2:
                    overlap = (ix2-ix1) * (iy2-iy1)
                    area_i = (x2-x1) * (y2-y1)
                    area_j = (boxes[j][2]-boxes[j][0]) * (boxes[j][3]-boxes[j][1])
                    if overlap / min(area_i, area_j) > overlap_threshold:
                        x1 = min(x1, boxes[j][0])
                        y1 = min(y1, boxes[j][1])
                        x2 = max(x2, boxes[j][2])
                        y2 = max(y2, boxes[j][3])
                        used[j] = True
            merged.append((x1, y1, x2, y2))
            used[i] = True

        return merged

    def detect(
        self,
        prev_frame: np.ndarray,
        curr_frame: np.ndarray,
        frame_idx: int = 0,
    ) -> Tuple[List[Dict], float, str]:
        # Safety net: full-frame detection every N frames
        if frame_idx % self.fallback_interval == 0:
            results = self.model(curr_frame, imgsz=self.imgsz,
                                conf=self.conf_threshold, iou=self.iou_threshold,
                                device=self.device, verbose=False)
            return self._extract_detections(results, offset=(0, 0)), 1.0, 'full'

        # Stage 1: Motion detection
        motion_boxes, motion_ratio = self.detect_motion_regions(prev_frame, curr_frame)

        # Skip detection if no significant motion
        if motion_ratio < self.motion_ratio_threshold or len(motion_boxes) == 0:
            return [], motion_ratio, 'skipped'

        # Stage 2: Run detection only on motion regions
        all_detections = []
        for x1, y1, x2, y2 in motion_boxes:
            crop = curr_frame[y1:y2, x1:x2]
            if crop.shape[0] < 10 or crop.shape[1] < 10:
                continue
            results = self.model(crop, imgsz=self.imgsz,
                                conf=self.conf_threshold, iou=self.iou_threshold,
                                device=self.device, verbose=False)
            all_detections.extend(self._extract_detections(results, offset=(x1, y1)))

        return all_detections, motion_ratio, 'gated'

    @staticmethod
    def _extract_detections(results, offset=(0, 0)) -> List[Dict]:
        """Extract detections from YOLO results and apply coordinate offset."""
        detections = []
        ox, oy = offset
        for r in results:
            if r.boxes is None:
                continue
            for box in r.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                detections.append({
                    'bbox': [x1 + ox, y1 + oy, x2 + ox, y2 + oy],
                    'confidence': box.conf[0].item(),
                    'class': int(box.cls[0].item()),
                })
        return detections

    def detect_video(
        self,
        video_path: str,
        output_path: Optional[str] = None,
    ) -> Dict:
        """
        Process a video with motion-gated detection.

        Returns dict with frame counts by mode, avg FPS, total detections.
        """
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        writer = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            writer = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

        prev_frame = None
        stats = {'total_frames': 0, 'skipped': 0, 'gated': 0, 'full': 0,
                 'total_detections': 0, 'inference_times': []}

        print(f"Processing: {video_path}")
        print(f"  {w}x{h} @ {fps:.1f}fps, {total_frames} frames")

        pbar = tqdm(total=total_frames, desc="Processing")
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame_idx = stats['total_frames']

            if prev_frame is None:
                results = self.model(frame, imgsz=self.imgsz,
                                    conf=self.conf_threshold, iou=self.iou_threshold,
                                    device=self.device, verbose=False)
                detections = self._extract_detections(results)
                mode = 'full'
            else:
                t_start = time.time()
                detections, _, mode = self.detect(prev_frame, frame, frame_idx)
                stats['inference_times'].append(time.time() - t_start)

            stats[mode] += 1
            stats['total_detections'] += len(detections)
            stats['total_frames'] += 1

            if writer:
                writer.write(self._draw_detections(frame, detections))

            prev_frame = frame.copy()
            pbar.update(1)

        cap.release()
        if writer:
            writer.release()
        pbar.close()

        if stats['inference_times']:
            times = np.array(stats['inference_times'])
            stats['avg_inference_time'] = float(times.mean())
            stats['avg_fps'] = float(1.0 / times.mean())
            stats['p95_inference_time'] = float(np.percentile(times, 95))

        self._print_stats(stats)
        return stats

    @staticmethod
    def _draw_detections(frame, detections, color=(0, 255, 0), thickness=2):
        annotated = frame.copy()
        for det in detections:
            x1, y1, x2, y2 = [int(v) for v in det['bbox']]
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, thickness)
            label = f"Drone: {det['confidence']:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(annotated, (x1, y1-th-5), (x1+tw, y1), color, -1)
            cv2.putText(annotated, label, (x1, y1-3),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
        return annotated

    @staticmethod
    def _print_stats(stats):
        total = stats['total_frames']
        print(f"\n{'='*50}")
        print(f"MOTION-GATED INFERENCE STATISTICS")
        print(f"{'='*50}")
        print(f"Total frames: {total}")
        print(f"  Skipped (no motion): {stats['skipped']} ({stats['skipped']/total*100:.1f}%)")
        print(f"  Gated (motion regions): {stats['gated']} ({stats['gated']/total*100:.1f}%)")
        print(f"  Full (fallback): {stats['full']} ({stats['full']/total*100:.1f}%)")
        print(f"Total detections: {stats['total_detections']}")
        if 'avg_fps' in stats:
            print(f"Average inference FPS: {stats['avg_fps']:.1f}")
            print(f"P95 latency: {stats['p95_inference_time']*1000:.1f}ms")

    def benchmark(self, video_path: str, compare_full_frame: bool = True) -> Dict:
        """
        Benchmark motion-gated vs full-frame detection.

        Produces the comparison table for the paper: FPS, latency, P95,
        energy proxy, detection count, and speedup factor.
        """
        print("=" * 60)
        print("BENCHMARKING: Motion-Gated vs Full-Frame Detection")
        print("=" * 60)

        print("\n--- Motion-Gated Detection ---")
        gated_stats = self.detect_video(video_path, output_path=None)

        if not compare_full_frame:
            return gated_stats

        print("\n--- Full-Frame Detection (Baseline) ---")
        cap = cv2.VideoCapture(video_path)
        full_times = []
        full_detections = 0
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        pbar = tqdm(total=total, desc="Full-frame baseline")
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            t_start = time.time()
            results = self.model(frame, imgsz=self.imgsz,
                                conf=self.conf_threshold, iou=self.iou_threshold,
                                device=self.device, verbose=False)
            full_times.append(time.time() - t_start)
            full_detections += len(self._extract_detections(results))
            pbar.update(1)
        cap.release()
        pbar.close()

        full_times = np.array(full_times)
        full_fps = 1.0 / full_times.mean()
        gated_fps = gated_stats.get('avg_fps', 0)

        # Energy proxy: assume typical ARM CPU power (Raspberry Pi 5: ~5W)
        power_w = 5.0

        print(f"\n{'='*60}")
        print(f"BENCHMARK RESULTS")
        print(f"{'='*60}")
        print(f"{'Metric':<30} {'Full-Frame':>15} {'Motion-Gated':>15}")
        print(f"{'-'*60}")
        print(f"{'Avg FPS':<30} {full_fps:>15.1f} {gated_fps:>15.1f}")
        print(f"{'Avg latency (ms)':<30} {full_times.mean()*1000:>15.1f} "
              f"{gated_stats.get('avg_inference_time',0)*1000:>15.1f}")
        print(f"{'P95 latency (ms)':<30} "
              f"{np.percentile(full_times,95)*1000:>15.1f} "
              f"{gated_stats.get('p95_inference_time',0)*1000:>15.1f}")
        print(f"{'Total detections':<30} {full_detections:>15d} "
              f"{gated_stats['total_detections']:>15d}")
        print(f"{'Energy/frame (mJ)':<30} "
              f"{full_times.mean()*power_w*1000:>15.1f} "
              f"{gated_stats.get('avg_inference_time',0)*power_w*1000:>15.1f}")
        print(f"{'Speedup':<30} {'1.00x':>15} {gated_fps/full_fps:>14.2f}x")

        return {
            'full_frame': {
                'fps': float(full_fps),
                'avg_latency_ms': float(full_times.mean() * 1000),
                'p95_latency_ms': float(np.percentile(full_times, 95) * 1000),
                'detections': full_detections,
            },
            'motion_gated': {
                'fps': float(gated_fps),
                'avg_latency_ms': float(gated_stats.get('avg_inference_time', 0) * 1000),
                'p95_latency_ms': float(gated_stats.get('p95_inference_time', 0) * 1000),
                'detections': gated_stats['total_detections'],
                'skipped_ratio': gated_stats['skipped'] / max(gated_stats['total_frames'], 1),
            },
            'speedup': float(gated_fps / full_fps) if full_fps > 0 else 0,
        }

### **Section 6: End-to-end Pipeline**

Run the complete pipeline end-to-end:
1. Two-dimensional sensitivity analysis (localization + discrimination)
2. Mixed-precision allocation optimizing combined objective
3. QAT training with small-target loss + confusion regularization
4. Export quantized model for edge deployment
5. Motion-gated inference benchmark

Args:
    confusion_classes: Pair of class names for discrimination analysis.
        Pass ['drone', 'bird'] to enable two-dimensional sensitivity
        analysis and confusion regularization. Pass None for
        localization-only analysis (backward-compatible).

In [12]:
# Imports
import os
import json

# Output Path
output_dir = 'results/'
os.makedirs(output_dir, exist_ok=True)

# Parameters
model_path = '/content/detect/train/weights/best.pt'
data_yaml = '/content/YOLO_Drone_Detector-1/data.yaml'
video_path = 'test.mp4'
target_size_ratio = 0.35
qat_epochs = 100
confusion_classes = ['bird', 'drone']

In [19]:
# Step 1: Sensitivity Analysis
print("=" * 60)
print("2D Quantization Sensitivity Analysis")
print("=" * 60)
analyzer = SensitivityAnalyzer(model_path, data_yaml, confusion_classes=confusion_classes)
sensitivity_path = os.path.join(output_dir, 'sensitivity_results.json')
sensitivity_results = analyzer.run_full_analysis(
    bits_list=[4, 8], output_path=sensitivity_path
)

2D Quantization Sensitivity Analysis
Found 88 Conv2d layers to analyze
Device: cuda
Discrimination tracking: bird vs drone
Measuring baseline (FP32) performance...
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1701.1±326.0 MB/s, size: 44.3 KB)
val: Scanning /content/YOLO_Drone_Detector-1/valid/labels.cache... 655 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 655/655 183.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 5.0it/s 8.1s
                   all        655       1442      0.815      0.806      0.841      0.427
Speed: 1.5ms preprocess, 4.0ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val-2
Baseline: mAP@0.5=0.8412, mAP=0.4274, AP_small=0.3723
  Per-class AP@0.5: {'bird': 0.72659434157

Analyzing model.0.conv @ 4bit:   0%|          | 0/176 [00:00<?, ?it/s]

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1770.1±882.8 MB/s, size: 43.4 KB)
val: Scanning /content/YOLO_Drone_Detector-1/valid/labels.cache... 655 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 655/655 228.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 4.4it/s 9.4s
                   all        655       1442      0.804      0.766      0.814      0.402
Speed: 2.2ms preprocess, 4.2ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val-3


Analyzing model.0.conv @ 8bit:   1%|          | 1/176 [00:12<35:48, 12.28s/it]

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1268.7±270.4 MB/s, size: 24.8 KB)
val: Scanning /content/YOLO_Drone_Detector-1/valid/labels.cache... 655 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 655/655 211.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 5.7it/s 7.2s
                   all        655       1442      0.816      0.805      0.841      0.426
Speed: 1.3ms preprocess, 4.2ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val-4


...

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1294.0±537.9 MB/s, size: 41.2 KB)
val: Scanning /content/YOLO_Drone_Detector-1/valid/labels.cache... 655 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 655/655 161.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 4.6it/s 8.8s
                   all        655       1442      0.437      0.551      0.484       0.22
Speed: 2.6ms preprocess, 4.2ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val-177


Analyzing model.23.dfl.conv @ 8bit:  99%|█████████▉| 175/176 [32:43<00:11, 11.22s/it]

Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1881.5±808.5 MB/s, size: 44.7 KB)
val: Scanning /content/YOLO_Drone_Detector-1/valid/labels.cache... 655 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 655/655 228.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 4.8it/s 8.6s
                   all        655       1442      0.802      0.798      0.834      0.422
Speed: 2.2ms preprocess, 4.2ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/runs/detect/val-178


Analyzing model.23.dfl.conv @ 8bit: 100%|██████████| 176/176 [32:54<00:00, 11.22s/it]


Results saved to results/sensitivity_results.json

SENSITIVITY ANALYSIS SUMMARY

4-bit quantization:
  Top 5 localization-sensitive layers (small-target AP drop):
    model.23.dfl.conv: 0.2028
    model.0.conv: 0.0321
    model.1.conv: 0.0202
    model.4.cv2.conv: 0.0173
    model.23.cv2.0.2: 0.0127
  Top 5 least localization-sensitive layers:
    model.10.m.0.attn.proj.conv: -0.0014
    model.10.m.0.attn.pe.conv: -0.0014
    model.23.cv2.0.1.conv: -0.0015
    model.8.m.0.m.0.cv2.conv: -0.0015
    model.2.cv1.conv: -0.0028
  Top 5 discrimination-sensitive layers (confusion increase):
    model.0.conv: 0.0066
    model.2.m.0.cv2.conv: 0.0058
    model.1.conv: 0.0044
    model.16.cv2.conv: 0.0044
    model.4.cv1.conv: 0.0029
  Top 5 least discrimination-sensitive layers:
    model.4.m.0.cv1.conv: -0.0015
    model.2.cv1.conv: -0.0017
    model.23.cv3.0.2: -0.0018
    model.23.cv3.0.0.1.conv: -0.0044
    model.23.dfl.conv: -0.0063
  Top 5 divergence layers (localization vs discrimination

In [20]:
# Step 2: Mixed-Precision Allocation
print("=" * 60)
print("Mixed-Precision Bit Allocation")
print("=" * 60)
allocator = MixedPrecisionAllocator(sensitivity_results, target_size_ratio=target_size_ratio)
allocation = allocator.dp_allocate()
allocation_path = os.path.join(output_dir, 'bit_allocation.json')
allocator.save_allocation(allocation, allocation_path)

Mixed-Precision Bit Allocation
Total layers: 88
FP32 model size: 9.82 MB
Target size (35%): 3.44 MB
Objective weights: w_loc=0.6, w_disc=0.4

DP allocation:
  Final size: 3.06 MB (31.2% of FP32)
  4-bit layers: 31
  8-bit layers: 27
  16-bit layers: 30
  Total localization sensitivity: -0.0318
  Total discrimination sensitivity: -0.0047
  Combined objective: -0.0209
Allocation saved to results/bit_allocation.json


In [21]:
from pathlib import Path

# Step 3: QAT Training
print("=" * 60)
print("QAT with Small-Target Loss + Confusion Regularization")
print("=" * 60)
trainer = QATTrainer(
    model_path=model_path,
    data_yaml=data_yaml,
    bit_allocation=allocation,
    epochs=qat_epochs
)
best_model = trainer.train()

QAT with Small-Target Loss + Confusion Regularization
QAT preparation: 58 layers quantized, 30 layers at full precision
QAT Trainer initialized:
  Epochs: 100
  Learning rate: 0.01
  Mixed precision: Yes
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/YOLO_Drone_Detector-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=Fal

In [29]:
# Step 4: Export Quantized Model
print("=" * 60)
print("Export Quantized Model for Edge Deployment")
print("=" * 60)

# Ensure best_model points to the correct absolute path
best_model = '/content/runs/detect/runs/qat/drone_qat/weights/best.pt'

# Re-instantiate the QATTrainer to pick up the latest class definition
# using parameters from previous cells.
# This is necessary because the QATTrainer class definition in 06d20e48 was updated
# after the original 'trainer' object was instantiated in facfc13d.
# The 'export_quantized' method internally reloads the model, so re-initializing
# the trainer here does not lose the trained weights.
trainer = QATTrainer(
    model_path=model_path, # Initial FP32 model path used for training setup
    data_yaml=data_yaml,
    bit_allocation=allocation,
    epochs=qat_epochs # The actual epochs value from the previous step
)

quantized_path = trainer.export_quantized(
    best_model, output_format='onnx',
    output_path=os.path.join(output_dir, 'model_quantized.onnx'),
)


Export Quantized Model for Edge Deployment
QAT preparation: 58 layers quantized, 30 layers at full precision
QAT Trainer initialized:
  Epochs: 100
  Learning rate: 0.01
  Mixed precision: Yes
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '/content/runs/detect/runs/qat/drone_qat/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.2 MB)

ONNX: starting export with onnx 1.22.0 opset 13...
ONNX: slimming with onnxslim 0.1.96...
ONNX: export success ✅ 1.4s, saved as '/content/runs/detect/runs/qat/drone_qat/weights/best.onnx' (10.1 MB)

Export complete (1.9s)
Results saved to /content/runs/detect/runs/qat/drone_qat/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/runs/qat/drone_qat/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/runs/d

In [ ]:
# Step 5: Motion-Gated Inference Benchmark
print("=" * 60)
print("Motion-Gated Inference Benchmark")
print("=" * 60)
detector = MotionGatedDetector(model_path=quantized_path, imgsz=640)
benchmark_results = detector.benchmark(video_path, compare_full_frame=True)

benchmark_path = os.path.join(output_dir, 'benchmark_results.json')
with open(benchmark_path, 'w') as f:
    json.dump(benchmark_results, f, indent=2, default=str)